# Crowdcal Phase 5: score the fine-tuned checkpoints on calib + test

This notebook runs `crowdcal run` for every `laya-ft` and `mbert-ce` checkpoint that the two Phase 4 notebooks produced. It writes only the cache JSONL files, which are small, so the 26 GB of checkpoints never have to leave Kaggle.

**Kaggle settings:**
- Accelerator: GPU T4 x2, with one checkpoint queue per GPU.
- Internet: on.
- Inputs: **Add Input → Notebooks → the outputs of `crowdcal-laya-ft` and `crowdcal-mbert-ce`**.

**Afterwards:** download `/kaggle/working/crowdcal_phase5.zip` and unzip it at the repo root. It fills `data/raw/<arm>@<size>-s<seed>/` and `data/checkpoints/<dir>/train_meta.json`.

The checkpoints were selected on dev before this runs (PREREG §5). This notebook computes no metrics.


In [ ]:
import glob, json, os, shutil, subprocess, sys
COMMIT = "c465425"   # crowdcal with LocalArm fixes (A1 unrounded laya, A3 max_length/device) + threaded run_arm
if not os.path.isdir("/kaggle/working/crowdcal"):
    subprocess.run(["git", "clone", "https://github.com/AmanSinghNp/crowdcal", "/kaggle/working/crowdcal"], check=True)
os.chdir("/kaggle/working/crowdcal")
subprocess.run(["git", "checkout", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps", "laya==0.3.21"], check=True)
assert subprocess.run(["git", "tag", "-l", "prereg-v1"], capture_output=True, text=True).stdout.strip(), "prereg tag missing"

# every selected checkpoint = a dir holding train_meta.json, from either notebook's output
ckpts = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train_meta.json", recursive=True)})
names = [os.path.basename(c) for c in ckpts]
print(len(ckpts), "checkpoints:", names)
assert len(ckpts) == 16 and len(set(names)) == 16, "expected 8 laya-ft + 8 mbert-ce checkpoints"
os.makedirs("data/checkpoints", exist_ok=True)
for c, n in zip(ckpts, names):
    dst = f"data/checkpoints/{n}"
    if not os.path.exists(dst):
        os.symlink(c, dst)  # config/arms.json resolves laya-ft / mbert-ce to data/checkpoints/<dir>


In [ ]:
# Warm the dataset cache once so the two GPU workers don't race on the downloads.
subprocess.run([sys.executable, "-c", "from crowdcal.data import load_split; load_split('calib'); load_split('test')"], check=True)

# Two queues, one per T4. Laya is the slow one (~15 min/checkpoint), so interleave the arms.
queues = [names[0::2], names[1::2]]
def worker(gpu, arms):
    cmds = " && ".join(f"crowdcal run --arm {a} --split calib && crowdcal run --arm {a} --split test" for a in arms)
    return subprocess.Popen(["bash", "-c", cmds], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
                            stdout=open(f"/kaggle/working/gpu{gpu}.log", "w"), stderr=subprocess.STDOUT)
procs = [worker(g, q) for g, q in enumerate(queues)]
codes = [p.wait() for p in procs]
for g in range(2):
    print(f"--- gpu{g} (exit {codes[g]}) ---"); print(open(f"/kaggle/working/gpu{g}.log").read()[-3000:])
assert codes == [0, 0], "a worker failed; rerun this cell (cached rows are skipped)"


In [ ]:
# Package: raw cache rows + each checkpoint's train_meta.json (no weights, no dataset text).
out = "/kaggle/working/phase5"
shutil.rmtree(out, ignore_errors=True)
for n in names:
    shutil.copytree(f"data/raw/{n}", f"{out}/data/raw/{n}")
    os.makedirs(f"{out}/data/checkpoints/{n}", exist_ok=True)
    shutil.copy(f"data/checkpoints/{n}/train_meta.json", f"{out}/data/checkpoints/{n}/train_meta.json")
shutil.make_archive("/kaggle/working/crowdcal_phase5", "zip", out)
rows = {n: sum(1 for f in glob.glob(f"data/raw/{n}/*.jsonl") for _ in open(f)) for n in names}
errs = {n: sum(json.loads(l)["status"] != "ok" for f in glob.glob(f"data/raw/{n}/*.jsonl") for l in open(f)) for n in names}
print(json.dumps({"rows": rows, "errors": errs}, indent=1))
print(os.path.getsize("/kaggle/working/crowdcal_phase5.zip") / 1e6, "MB")
shutil.rmtree("/kaggle/working/crowdcal")  # keep Output small: only the zip + logs
